# 01 — compound intersections + manifest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cellduet/blob/main/notebooks/01_intersections.ipynb)

The first executable gate of v0. Verifies the InChIKey overlaps that the rest of the plan depends on:

| Arm | Compound source A | Compound source B | Target | Join key |
|---|---|---|---|---|
| **B3b primary** | Tahoe-100M (`drug_metadata`) | JUMP-CP cpg0016 (`compound.csv.gz`) | **228** | full InChIKey |
| **B3a robustness** | Tahoe-100M | rxrx3-core (`metadata_rxrx3_core.csv`) | **145** | skeleton InChIKey |
| **B3c sanity** | Tahoe-100M | JUMP-Target-1 (`JUMP-Target-1_compound_metadata.tsv`) | **14** (11 trt + 3 poscon) | full InChIKey |

Builds and persists the joint compound manifest (`compound_manifest.parquet`) used by every downstream notebook. Optionally pushes it to HF as `patrickjreed/cellduet-compound-manifest`.

Refs: `docs/datasets/joint.md`, per-dataset dossiers in `docs/datasets/`.

## 1. Install + imports

In [ ]:
!pip install -q pandas pyarrow huggingface_hub rdkit s3fs
!pip install -q --no-deps "git+https://github.com/PatrickJReed/cellduet.git@main"

In [ ]:
from pathlib import Path

import pandas as pd
import s3fs
from huggingface_hub import HfApi, hf_hub_download, login

from cellduet.inchikey import compute_inchikey

print("imports OK")

## 2. HF login + cache dir

`HF_TOKEN` Colab secret with **write** scope is required if you want the optional HF push at the end. Read-only fetches work without auth.

In [ ]:
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("HF login: OK (Colab secret)")
except Exception as e:
    print(f"Colab secret not used ({type(e).__name__}); relying on local cache or anon HF reads")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/cellduet/cache")
except (ImportError, Exception) as e:
    print(f"Drive skipped ({type(e).__name__}); using local cache")
    CACHE = Path.home() / ".cache" / "cellduet"
CACHE.mkdir(parents=True, exist_ok=True)
print(f"cache: {CACHE}")

## 3. Tahoe-100M drug metadata

Pulls the 379-row `drug_metadata.parquet` from `tahoebio/Tahoe-100M` (CC0-1.0). Computes full + skeleton InChIKey from `canonical_smiles`.

In [ ]:
tahoe_path = hf_hub_download(
    "tahoebio/Tahoe-100M", "metadata/drug_metadata.parquet", repo_type="dataset"
)
tahoe = pd.read_parquet(tahoe_path)
tahoe["inchikey_full"] = tahoe["canonical_smiles"].apply(
    lambda s: compute_inchikey(s, "full") if isinstance(s, str) else None
)
tahoe["inchikey_skeleton"] = tahoe["inchikey_full"].apply(
    lambda k: k[:14] if isinstance(k, str) else None
)
n_parsed = tahoe["inchikey_full"].notna().sum()
print(f"Tahoe drugs: {len(tahoe)}, {n_parsed} with parseable canonical_smiles")

## 4. JUMP cpg0016 compound metadata

The 115,795-compound master metadata (`Metadata_JCP2022` -> `Metadata_InChIKey`), pulled from the `jump-cellpainting/datasets` GitHub mirror (CC0-1.0). The cpg0016 harmony-corrected feature parquet is keyed on `Metadata_JCP2022`; this CSV provides the InChIKey lookup.

In [ ]:
jump_url = "https://github.com/jump-cellpainting/datasets/raw/main/metadata/compound.csv.gz"
jump = pd.read_csv(jump_url)
jump_keys_full = set(jump["Metadata_InChIKey"].dropna())
print(f"JUMP cpg0016 compounds: {len(jump)} rows, {len(jump_keys_full)} unique InChIKeys")

## 5. rxrx3-core metadata

The 222,601-row `metadata_rxrx3_core.csv` from `recursionpharma/rxrx3-core` (Recursion bespoke EULA; CC-BY-SA-like with neuroscience carve-out). Filter to COMPOUND wells, dedupe on SMILES, compute skeleton InChIKey.

In [ ]:
rxrx3_path = hf_hub_download(
    "recursionpharma/rxrx3-core", "metadata_rxrx3_core.csv", repo_type="dataset"
)
rxrx3 = pd.read_csv(rxrx3_path, low_memory=False)
rxrx3_compounds = (
    rxrx3[rxrx3["perturbation_type"] == "COMPOUND"]
    .dropna(subset=["SMILES"])
    .drop_duplicates(subset=["SMILES"])
    .copy()
)
rxrx3_compounds["inchikey_skeleton"] = rxrx3_compounds["SMILES"].apply(
    lambda s: compute_inchikey(s, "skeleton")
)
rxrx3_skel = set(rxrx3_compounds["inchikey_skeleton"].dropna())
print(
    f"rxrx3-core: {rxrx3.shape[0]} total wells, "
    f"{len(rxrx3_compounds)} unique compounds, "
    f"{len(rxrx3_skel)} unique skeleton InChIKeys"
)

## 6. JUMP-Target-1 compound platemap (for B3c)

CPJUMP1 (cpg0000-jump-pilot) used the JUMP-Target-1 plate layout: 302 unique compounds (260 treatments + 46 positive controls + 1 negative control + variants). The canonical metadata lives at `jump-cellpainting/JUMP-Target` on GitHub (CC0). The cpg0000 harmonized metadata draft on S3 only includes wells that were imaged in the canonical batch and is sometimes a strict subset of the platemap; the GitHub TSV is the authoritative compound list.

In [ ]:
jt1_url = (
    "https://raw.githubusercontent.com/jump-cellpainting/JUMP-Target/master/"
    "JUMP-Target-1_compound_metadata.tsv"
)
jt1 = pd.read_csv(jt1_url, sep="\t")
jt1_keys_full = set(jt1["InChIKey"].dropna())
poscon_mask = jt1["control_type"].astype(str).str.contains("poscon", case=False, na=False)
jt1_poscon_keys = set(jt1.loc[poscon_mask, "InChIKey"].dropna())
print(
    f"JUMP-Target-1 platemap: {len(jt1)} rows, {len(jt1_keys_full)} unique InChIKeys, "
    f"{len(jt1_poscon_keys)} positive-control InChIKeys"
)

## 7. Compute the three intersections

In [ ]:
tahoe_full = set(tahoe["inchikey_full"].dropna())
tahoe_skel = set(tahoe["inchikey_skeleton"].dropna())

b3b_overlap = tahoe_full & jump_keys_full
b3a_overlap = tahoe_skel & rxrx3_skel
b3c_overlap = tahoe_full & jt1_keys_full
b3c_treatments = b3c_overlap - jt1_poscon_keys
b3c_poscon = b3c_overlap & jt1_poscon_keys

print(f"B3b primary    Tahoe ∩ JUMP cpg0016  (full):     {len(b3b_overlap):>4d}  target=228")
print(f"B3a robustness Tahoe ∩ rxrx3-core    (skeleton): {len(b3a_overlap):>4d}  target=145")
print(f"B3c sanity     Tahoe ∩ JUMP-Target-1 (full):     {len(b3c_overlap):>4d}  target=14")
print(f"               of which positive controls:       {len(b3c_poscon):>4d}  target=3")
print(f"               of which treatment compounds:     {len(b3c_treatments):>4d}  target=11")

## 8. Verify gates

In [ ]:
assert 220 <= len(b3b_overlap) <= 240, (
    f"B3b overlap {len(b3b_overlap)} outside the 220-240 acceptance window"
)
assert 130 <= len(b3a_overlap) <= 160, (
    f"B3a overlap {len(b3a_overlap)} outside the 130-160 acceptance window"
)
assert len(b3c_treatments) >= 10, (
    f"B3c treatment-only overlap {len(b3c_treatments)} below the 10-compound floor"
)
print("All three v0 gates clear.")

## 9. Build compound manifest

One row per Tahoe drug, with boolean membership flags per arm and the looked-up keys. Saved to Drive cache.

In [ ]:
manifest = tahoe[["drug", "canonical_smiles", "inchikey_full", "inchikey_skeleton"]].copy()
manifest["in_tahoe"] = True
manifest["in_jump_cpg0016"] = manifest["inchikey_full"].isin(b3b_overlap)
manifest["in_rxrx3_core"] = manifest["inchikey_skeleton"].isin(b3a_overlap)
manifest["in_jt1_treatment"] = manifest["inchikey_full"].isin(b3c_treatments)
manifest["in_jt1_poscon"] = manifest["inchikey_full"].isin(b3c_poscon)

manifest_path = CACHE / "compound_manifest.parquet"
manifest.to_parquet(manifest_path)
print(f"manifest saved: {manifest_path}  ({manifest_path.stat().st_size / 1024:.1f} KB)")
print()
print(manifest[
    ["drug", "in_jump_cpg0016", "in_rxrx3_core", "in_jt1_treatment", "in_jt1_poscon"]
].head(10).to_string(index=False))

## 10. (optional) push manifest to HF

Skips silently if you don't have a write token configured. To run, add `HF_TOKEN` as a Colab secret with write scope and re-execute.

In [ ]:
try:
    api = HfApi()
    me = api.whoami()
    repo_id = "patrickjreed/cellduet-compound-manifest"
    api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(manifest_path),
        path_in_repo="compound_manifest.parquet",
        repo_id=repo_id,
        repo_type="dataset",
    )
    print(f"pushed manifest to HF: {repo_id}")
except Exception as e:
    print(f"HF push skipped ({type(e).__name__}): {e}")

## 11. Summary

In [ ]:
print("v0 compound-manifest summary")
print("-" * 40)
print(f"B3b primary    {len(b3b_overlap):>4d} compounds  Tahoe x JUMP cpg0016")
print(f"B3a robustness {len(b3a_overlap):>4d} compounds  Tahoe x rxrx3-core")
print(f"B3c sanity     {len(b3c_treatments):>4d} treatments + {len(b3c_poscon)} poscon  Tahoe-A549 x CPJUMP1-A549")
print()
print("All three gates clear -> v0 is feasible.")